[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/CBS_Third_Generation.ipynb)


# Occupational coding as evidence reconciliation
### Independent third-generation solution · CBS simulated survey

**Decision:** treat a record as a collection of partially independent claims about a person's occupation. Separate information sufficiency, conflicting evidence, label reliability, and predictive generalization. This notebook obtains the immutable original workbook directly from the public repository and runs top-to-bottom in Google Colab (CPU).

In [ ]:
import io, hashlib, urllib.request, re, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings('ignore',category=UserWarning)
SEED=2026
URL='https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx'
SHA='e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac'
raw=urllib.request.urlopen(URL,timeout=90).read()
assert hashlib.sha256(raw).hexdigest()==SHA, 'Original workbook hash mismatch'
book=pd.ExcelFile(io.BytesIO(raw)); print('Sheets:',book.sheet_names, 'Verified bytes:',len(raw))

In [ ]:
data=pd.read_excel(io.BytesIO(raw),sheet_name='Survey Dataset',usecols='A:R',dtype=str)
metadata=pd.read_excel(io.BytesIO(raw),sheet_name='Metadata',usecols='A:D',dtype=str)
data.columns=data.columns.astype(str).str.strip(); metadata.columns=metadata.columns.astype(str).str.strip()
data=data.loc[data['ID'].notna()].copy().reset_index(drop=True)
assert len(data)==200 and data['ID'].nunique()==200 and 'SemelMishlachSofi' in data
assert len(data.columns)==18
original=data.copy(deep=True)
print('Survey:',data.shape,'Metadata:',metadata.shape)
display(metadata.head(20));display(data.head(3))

## Inference 1 — Three different kinds of uncertainty
A respondent may omit information; a respondent may provide mutually inconsistent descriptions; a human coder may output an incomplete code. These phenomena should be quantified separately, not collapsed into a single missingness indicator.

In [ ]:
TEXT=['YeshuvAvoda','ShemAvoda','SugAvoda','ShemMachlaka','SugMachlaka','EzoAvoda','TeurPeula','TeurTafkid']
OCC=['SugAvoda','ShemMachlaka','SugMachlaka','EzoAvoda','TeurPeula','TeurTafkid']
TARGET='SemelMishlachSofi'
def clean(x):
    if pd.isna(x):return ''
    return re.sub(r'\s+',' ',str(x).strip()).casefold()
def meaningful(x):return bool(clean(x)) and clean(x) not in {'כנ"ל','כנ״ל','כנל','חסוי','לא יכול לפרט'}
for c in TEXT:data[c+'_observed']=data[c].map(meaningful).astype(int)
data['occupation_evidence_count']=data[[c+'_observed' for c in OCC]].sum(axis=1)
data['target_resolution']=data[TARGET].fillna('').map(lambda x:sum(ch!='X' for ch in str(x)) if len(str(x))==4 else 0)
data['target_complete']=data[TARGET].fillna('').str.fullmatch(r'\d{4}').fillna(False)
display(data.groupby('occupation_evidence_count',dropna=False).agg(records=('ID','size'),fully_coded=('target_complete','sum'),mean_code_resolution=('target_resolution','mean')))
print('X-coded:',int((~data.target_complete).sum()))

## Inference 2 — A record's evidentiary identity differs from its administrative identity
`ID` distinguishes rows but does not encode occupation. Two rows with different IDs can repeat the same substantive account. Construct evidence signatures and compare exact repetition, conflicting targets, and similarity without ever feeding ID to a classifier.

In [ ]:
def signature(row,cols):return ' | '.join(clean(row[c]) for c in cols)
data['evidence_signature']=data.apply(lambda r:signature(r,OCC),axis=1)
data['all_text_signature']=data.apply(lambda r:signature(r,TEXT),axis=1)
rep=data.groupby('evidence_signature').agg(n=('ID','size'),targets=(TARGET,lambda s:sorted(set(s.dropna()))),target_count=(TARGET,'nunique')).sort_values('n',ascending=False)
print('Repeated occupation descriptions:',int((rep.n>1).sum()))
print('Conflicting codes for identical occupation descriptions:',int(((rep.n>1)&(rep.target_count>1)).sum()))
display(rep[(rep.n>1)&(rep.target_count>1)].head(12))

## Inference 3 — Occupation is primarily about performed work, not the employer's industry
A firm can employ software engineers, cleaners, accountants and managers. Represent role evidence and organizational context separately, then test their marginal predictive contributions. This is an empirical ablation, not an assertion that either source is always irrelevant.

In [ ]:
ROLE=['EzoAvoda','TeurPeula','TeurTafkid','MenahelEtMi']
CONTEXT=['ShemAvoda','SugAvoda','ShemMachlaka','SugMachlaka','YeshuvAvoda']
OTHER=['MaamadAvoda','MakorSachar','TeudaGvoha','shnotlimud','Gil','TarichHathala']
def pack(frame,cols):
    return frame.apply(lambda r:' '.join(f'{c} {clean(r[c])}' for c in cols if meaningful(r[c])),axis=1)
for name,cols in [('role',ROLE),('context',CONTEXT),('combined',ROLE+CONTEXT+OTHER)]:data['text_'+name]=pack(data,cols)
valid=data.loc[data.target_complete].copy()
print('Fully specified target rows:',len(valid),'classes:',valid[TARGET].nunique())
print('Top target frequencies:');display(valid[TARGET].value_counts().head(12))

## Evaluation: prevent identical descriptions appearing in training and test
Group by substantive occupation descriptions. Since a 200-row sample can have rare labels, report the attainable held-out scores without implying population validity. Compare a majority baseline with independent text models; use a deterministic GroupKFold and explicitly report unseen labels.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
models={
 'majority':DummyClassifier(strategy='most_frequent'),
 'word_tfidf_logreg':make_pipeline(TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1),LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED)),
 'char_tfidf_svm':make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,5),min_df=1,max_features=25000),LinearSVC(class_weight='balanced',random_state=SEED)),
 'char_tfidf_nb':make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,4),min_df=1,max_features=25000),ComplementNB(alpha=0.5))}
groups=valid['evidence_signature']; folds=min(5,groups.nunique())
assert folds>=2
cv=GroupKFold(n_splits=folds)
results=[]; predictions=[]
for view in ['role','context','combined']:
    X=valid['text_'+view].reset_index(drop=True);y=valid[TARGET].reset_index(drop=True);g=groups.reset_index(drop=True)
    for name,model in models.items():
        yp=np.empty(len(y),dtype=object);unseen=[]
        for train,test in cv.split(X,y,g):
            fitted=clone(model);fitted.fit(X.iloc[train],y.iloc[train]);yp[test]=fitted.predict(X.iloc[test]);unseen.extend([int(i) for i in test if y.iloc[i] not in set(y.iloc[train])])
        score={'view':view,'model':name,'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro',zero_division=0),'unseen_label_rows':len(unseen)}
        results.append(score)
        predictions.append(pd.DataFrame({'ID':valid.ID.to_numpy(),'view':view,'model':name,'actual':y,'predicted':yp,'correct':y.to_numpy()==yp}))
results=pd.DataFrame(results).sort_values(['macro_f1','accuracy'],ascending=False)
display(results)
print('Best measured configuration:',results.iloc[0].to_dict())

## Case review and operational interpretation
A model prediction is a proposal, not an authoritative recoding. Prioritize cases with conflicting duplicate evidence, incomplete human labels, and absent substantive descriptions. Preserve all original fields and expose the reasons for referral.

In [ ]:
best=results.iloc[0];best_preds=next(p for p in predictions if p['view'].iloc[0]==best['view'] and p['model'].iloc[0]==best['model'])
review=data[['ID',TARGET,'occupation_evidence_count','target_resolution','evidence_signature']].copy()
review['conflicting_evidence']=review['evidence_signature'].map(rep.target_count).gt(1)
review['needs_human_review']=(~data.target_complete)|(data.occupation_evidence_count<2)|review.conflicting_evidence
review=review.merge(best_preds[['ID','predicted','correct']],on='ID',how='left')
display(review.sort_values(['needs_human_review','occupation_evidence_count'],ascending=[False,True]).head(25))
print('Review queue:',int(review.needs_human_review.sum()),'of',len(review))
assert original.shape==(200,18) and original.equals(data[original.columns])
print('SOURCE PRESERVED: all 200 original rows and 18 original columns unchanged')

In [ ]:
output=Path('cbs_third_generation_results');output.mkdir(exist_ok=True)
results.to_csv(output/'model_ablation.csv',index=False)
review.to_csv(output/'human_review_queue.csv',index=False,encoding='utf-8-sig')
pd.concat(predictions,ignore_index=True).to_csv(output/'held_out_predictions.csv',index=False)
manifest={'source_sha256':SHA,'rows':len(original),'columns':len(original.columns),'full_label_rows':len(valid),'cv_folds':folds,'best':best.to_dict()}
(output/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Results saved:',[p.name for p in output.iterdir()])

## Conclusions and deployment boundary
1. **Identity:** `ID` is an administrative key; repeated occupation evidence is a distinct analytical phenomenon.
2. **Uncertainty:** distinguish respondent nonresponse, non-informative wording, evidence conflict, and incomplete human classification.
3. **Causality-aware features:** compare job duties against organizational context instead of assuming employer identity determines occupation.
4. **Evaluation:** grouped holdouts measure transfer across descriptions; rare/unseen labels limit what this small simulation can establish.
5. **Deployment:** present predictions with an auditable review queue, preserve original responses, and request human adjudication when the evidence cannot justify full specificity.

All numerical claims above are computed by the executable cells; no experimental score is pre-filled.